###Imports

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
    EarlyStoppingCallback,
)


###data


In [ ]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

In [ ]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"]
    df["job_no"] = df["job_no"]
    return df

In [ ]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content/linkedin-cvs-not-annotated.json")

#Approach

### Pretrained transformer fine-tuning with XLM-R

- Model setup:
  - Base model: xlm-roberta-base
  - Tokenization: AutoTokenizer, truncation with MAX_LEN = 48, dynamic padding via DataCollatorWithPadding
  - Training: HuggingFace Trainer + TrainingArguments
  - Metrics tracked: accuracy and macro F1

### Department models
- Labels built from department-v2 and mapped to ids (label2id, id2label)
- Training variants:
  - Train/validation split on department-v2 (stratified split, evaluated per epoch)
  - Training on full department-v2
- Experiment to improve "Other" recall:
  - Two-stage pipeline:
    - Stage 1: "Other" vs Not-"Other"
    - Stage 2: classifier on labels excluding "Other"
    - Final prediction: if Stage 1 predicts "Other", output "Other", otherwise output Stage 2 label

### Seniority models
- Labels built from seniority-v2 and mapped to ids (label2id, id2label)
- Training: stratified train/validation split on seniority-v2
- Inference-time idea for ambiguous titles: Thresholding to predict unseen label "Professional":
    - compute max softmax probability per sample
    - if confidence is below a threshold (0.55), overwrite prediction to "Professional"


### Evaluation outputs
For each run, I reported:
- accuracy
- macro F1
- precision / recall / F1 per class (classification report)
- confusion matrix

#Results


### Department
- Department-v2 internal validation (train/val split):
  - final validation accuracy: 0.9867, Recall avg: 0.7668
- LinkedIn evaluation (trained on department-v2 split model 80/20, 2 epochs):
  - accuracy: 0.3988, Recall avg: 0.4770
- LinkedIn evaluation (trained on full department-v2, 2 epochs):
  - accuracy: 0.3707, Recall avg:  0.5109
- Two-stage "Other" vs Rest experiment:
  - Overall combined accuracy: 0.3465, Recall avg: 0.50

### Seniority
- Seniority-v2 internal validation (train/val split):
  - validation accuracy: 0.9905, Recall avg: 0.9847
- LinkedIn evaluation (trained on department-v2 split model 90/10, 3 epochs):
  - accuracy: 0.3772, Recall avg: 0.6189
- LinkedIn evaluation (thresholding to Professional with 0.55):
  - accuracy: 0.3901, Recall avg: 0.6233

#Models

##Department models

####train/Test split - department-v2

In [ ]:
MODEL = "xlm-roberta-base"
MAX_LEN = 48
BATCH = 32
EPOCHS = 2
set_seed(42)

# --- Labels ---
dept_labels = sorted(df_department["label"].unique().tolist())
dept_label2id = {l: i for i, l in enumerate(dept_labels)}
dept_id2label = {i: l for l, i in dept_label2id.items()}

# --- Trainings/eval data (label list) ---
df_dept = df_department.copy()
df_dept["labels"] = df_dept["label"].map(dept_label2id)

train_df, eval_df = train_test_split(
    df_dept[["text", "labels"]],
    test_size=0.20,
    random_state=42,
    stratify=df_dept["labels"],
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds   = Dataset.from_pandas(eval_df.reset_index(drop=True))

# --- Tokenize  ---
dept_tokenizer = AutoTokenizer.from_pretrained(MODEL)
dept_collator = DataCollatorWithPadding(tokenizer=dept_tokenizer)

def tokenize(batch):
    return dept_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_ds  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

# --- Metrics ---
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

# --- Model ---
dept_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(dept_labels),
    label2id=dept_label2id,
    id2label=dept_id2label,
)

dept_args = TrainingArguments(
    output_dir="dept_model_xlm-r_2",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

dept_trainer_model_xlmr_2 = Trainer(
    model=dept_model,
    args=dept_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=dept_tokenizer,
    data_collator=dept_collator,
    compute_metrics=compute_metrics,
)

dept_trainer_model_xlmr_2.train()

Map:   0%|          | 0/8116 [00:00<?, ? examples/s]

Map:   0%|          | 0/2029 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.126941,0.976343,0.680577
2,0.272000,0.064241,0.986693,0.766801


TrainOutput(global_step=508, training_loss=0.26893623818562723, metrics={'train_runtime': 99.8422, 'train_samples_per_second': 162.577, 'train_steps_per_second': 5.088, 'total_flos': 140177574543336.0, 'train_loss': 0.26893623818562723, 'epoch': 2.0})

In [ ]:
# --- predict ---
ext = df_t_annotated[["position", "department"]].copy()
ext["text"] = ext["position"]
ext["labels"] = ext["department"].map(dept_label2id).astype(int)

ext_ds = Dataset.from_pandas(ext[["text", "labels"]].reset_index(drop=True), preserve_index=False)
ext_ds = ext_ds.map(tokenize, batched=True, remove_columns=["text"])

pred = dept_trainer_model_xlmr_2.predict(ext_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)

print("accuracy:", round((y_pred == y_true).mean(), 4))

labels = sorted(set(y_true) | set(y_pred))
target_names = [dept_id2label[int(i)] for i in labels]

print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(target_names, name="true"), columns=pd.Index(target_names, name="pred"))
cm_df


Map:   0%|          | 0/2638 [00:00<?, ? examples/s]

accuracy: 0.3988
                        precision    recall  f1-score   support

        Administrative     0.0641    0.3571    0.1087        84
  Business Development     0.2372    0.4744    0.3162        78
            Consulting     0.3394    0.6718    0.4509       195
      Customer Support     0.4857    0.3542    0.4096        48
       Human Resources     0.0000    0.0000    0.0000        70
Information Technology     0.5231    0.6538    0.5812       312
             Marketing     0.5286    0.5564    0.5421       133
                 Other     0.5767    0.1861    0.2814      1252
    Project Management     0.5234    0.7029    0.6000       175
            Purchasing     0.2108    0.5417    0.3035        72
                 Sales     0.6862    0.7489    0.7162       219

              accuracy                         0.3988      2638
             macro avg     0.3796    0.4770    0.3918      2638
          weighted avg     0.5027    0.3988    0.3908      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,30,3,3,1,0,10,1,20,4,10,2
Business Development,2,37,1,0,0,4,1,9,14,7,3
Consulting,10,18,131,0,0,5,5,11,7,6,2
Customer Support,2,3,1,17,0,3,1,16,1,0,4
Human Resources,10,1,11,4,0,1,3,37,0,1,2
Information Technology,20,8,21,5,0,204,3,35,8,2,6
Marketing,13,1,9,1,0,19,74,6,5,3,2
Other,361,81,197,6,0,125,37,233,69,91,52
Project Management,4,4,6,0,0,13,3,11,123,9,2


####training on whole department_v2

In [ ]:
MODEL = "xlm-roberta-base"
MAX_LEN = 48
BATCH = 32
EPOCHS = 2

# --- Labels ---
train_labels = df_department["label"].unique().tolist()
eval_labels  = df_t_annotated["department"].unique().tolist()

dept_labels = sorted(set(train_labels) | set(eval_labels))
dept_label2id = {l:i for i,l in enumerate(dept_labels)}
dept_id2label = {i:l for l,i in dept_label2id.items()}

# --- Train dataset (label list) ---
train_df = df_department.copy()
train_df["text"] = train_df["text"]
train_df["labels"] = train_df["label"].map(dept_label2id)
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)

# --- Eval dataset (LinkedIn annotated) ---
eval_df = df_t_annotated.copy()
eval_df["text"] = eval_df["position"]
eval_df["labels"] = eval_df["department"].map(dept_label2id)
eval_ds = Dataset.from_pandas(eval_df[["text", "labels"]], preserve_index=False)

# --- Tokenize  ---
dept_tokenizer = AutoTokenizer.from_pretrained(MODEL)
dept_collator = DataCollatorWithPadding(tokenizer=dept_tokenizer)

def tokenize(batch):
    return dept_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_ds  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

# --- Metrics ---
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

# --- Model ---
dept_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(dept_labels),
    id2label=dept_id2label,
    label2id=dept_label2id,
)

training_args = TrainingArguments(
    output_dir="dept_model_xlm-r_3",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    logging_steps=100,
    seed=42,
)

trainer_dept_model_xlmr_3 = Trainer(
    model=dept_model,
    args=training_args,
    train_dataset=train_ds,
    #eval_dataset=eval_ds,
    tokenizer=dept_tokenizer,
    data_collator=dept_collator,
    compute_metrics=compute_metrics,
)

trainer_dept_model_xlmr_3.train()

Map:   0%|          | 0/10145 [00:00<?, ? examples/s]

Map:   0%|          | 0/2638 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,0.788000
200,0.247800
300,0.157700
400,0.095000
500,0.079600
600,0.055200


TrainOutput(global_step=636, training_loss=0.22582803226117068, metrics={'train_runtime': 107.1878, 'train_samples_per_second': 189.294, 'train_steps_per_second': 5.934, 'total_flos': 175255373649060.0, 'train_loss': 0.22582803226117068, 'epoch': 2.0})

In [ ]:
# --- predict ---
pred = trainer_dept_model_xlmr_3.predict(eval_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)


print("accuracy:", round((y_pred == y_true).mean(), 4))

labels = sorted(set(y_true) | set(y_pred))
target_names = [dept_id2label[int(i)] for i in labels]

print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(target_names, name="true"), columns=pd.Index(target_names, name="pred"))
cm_df


accuracy: 0.3707
                        precision    recall  f1-score   support

        Administrative     0.1719    0.1310    0.1486        84
  Business Development     0.2836    0.4872    0.3585        78
            Consulting     0.2851    0.6769    0.4012       195
      Customer Support     0.9000    0.1875    0.3103        48
       Human Resources     0.5057    0.6286    0.5605        70
Information Technology     0.2392    0.8846    0.3765       312
             Marketing     0.5804    0.4887    0.5306       133
                 Other     0.7889    0.0567    0.1058      1252
    Project Management     0.6158    0.6686    0.6411       175
            Purchasing     0.4182    0.6389    0.5055        72
                 Sales     0.7545    0.7717    0.7630       219

              accuracy                         0.3707      2638
             macro avg     0.5039    0.5109    0.4274      2638
          weighted avg     0.6116    0.3707    0.3067      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,11,2,1,0,1,48,1,9,1,9,1
Business Development,3,38,8,0,0,21,2,1,0,2,3
Consulting,2,19,132,0,5,26,4,2,4,0,1
Customer Support,2,3,2,9,0,28,0,0,0,0,4
Human Resources,0,0,11,0,44,9,2,2,0,1,1
Information Technology,2,7,10,1,4,276,0,2,9,0,1
Marketing,2,1,8,0,0,48,65,1,1,3,4
Other,39,56,283,0,30,616,31,71,54,36,36
Project Management,0,4,4,0,1,43,3,0,117,0,3


####Two Stages

#####Stage 1

In [ ]:
MODEL = "xlm-roberta-base"
MAX_LEN = 48
BATCH = 32
EPOCHS = 4
OTHER = "Other"
set_seed(42)


# Stage 1: Other vs Rest

# --- Labels ---
stage1_labels = ["Not-Other", "Other"]
stage1_label2id = {l: i for i, l in enumerate(stage1_labels)}
stage1_id2label = {i: l for l, i in stage1_label2id.items()}

# --- Training/eval data ---
df_s1 = df_department[["text", "label"]].copy()
df_s1["labels"] = (df_s1["label"] == OTHER).astype(int)

s1_train_df, s1_eval_df = train_test_split(
    df_s1[["text", "labels"]],
    test_size=0.10,
    random_state=42,
    stratify=df_s1["labels"],
)

s1_train_ds = Dataset.from_pandas(s1_train_df.reset_index(drop=True))
s1_eval_ds  = Dataset.from_pandas(s1_eval_df.reset_index(drop=True))

# --- Tokenize  ---
s1_tokenizer = AutoTokenizer.from_pretrained(MODEL)
s1_collator = DataCollatorWithPadding(tokenizer=s1_tokenizer)

def s1_tokenize(batch):
    return s1_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

s1_train_ds = s1_train_ds.map(s1_tokenize, batched=True, remove_columns=["text"])
s1_eval_ds  = s1_eval_ds.map(s1_tokenize, batched=True, remove_columns=["text"])

# --- Metrics ---
def s1_compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_other": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

# --- Model ---
stage1_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(stage1_labels),
    label2id=stage1_label2id,
    id2label=stage1_id2label,
)

stage1_args = TrainingArguments(
    output_dir="stage1_other_xlmr",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

stage1_trainer = Trainer(
    model=stage1_model,
    args=stage1_args,
    train_dataset=s1_train_ds,
    eval_dataset=s1_eval_ds,
    tokenizer=s1_tokenizer,
    data_collator=s1_collator,
    compute_metrics=s1_compute_metrics,
)

stage1_trainer.train()





Map:   0%|          | 0/9130 [00:00<?, ? examples/s]

Map:   0%|          | 0/1015 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Other
1,No log,0.012844,0.996059,0.000000
2,0.026700,0.009122,0.999015,0.857143
3,0.026700,0.009415,0.999015,0.857143
4,0.003700,0.003181,0.999015,0.857143


TrainOutput(global_step=1144, training_loss=0.013586380733893467, metrics={'train_runtime': 234.3735, 'train_samples_per_second': 155.82, 'train_steps_per_second': 4.881, 'total_flos': 320402459887080.0, 'train_loss': 0.013586380733893467, 'epoch': 4.0})

#####Stage 2

In [ ]:

# Stage 2: train on department-v2 where label != Other

# --- Labels ---
df_s2 = df_department[df_department["label"] != OTHER][["text", "label"]].copy()

dept_labels = sorted(df_s2["label"].unique().tolist())
dept_label2id = {l: i for i, l in enumerate(dept_labels)}
dept_id2label = {i: l for l, i in dept_label2id.items()}

df_s2["labels"] = df_s2["label"].map(dept_label2id)

# --- Training/eval data ---
s2_train_df, s2_eval_df = train_test_split(
    df_s2[["text", "labels"]],
    test_size=0.10,
    random_state=42,
    stratify=df_s2["labels"],
)

s2_train_ds = Dataset.from_pandas(s2_train_df.reset_index(drop=True))
s2_eval_ds  = Dataset.from_pandas(s2_eval_df.reset_index(drop=True))

# --- Tokenize  ---
s2_tokenizer = AutoTokenizer.from_pretrained(MODEL)
s2_collator = DataCollatorWithPadding(tokenizer=s2_tokenizer)

def s2_tokenize(batch):
    return s2_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

s2_train_ds = s2_train_ds.map(s2_tokenize, batched=True, remove_columns=["text"])
s2_eval_ds  = s2_eval_ds.map(s2_tokenize, batched=True, remove_columns=["text"])

# --- Metrics ---
def s2_compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# --- Model ---
stage2_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(dept_labels),
    label2id=dept_label2id,
    id2label=dept_id2label,
)

stage2_args = TrainingArguments(
    output_dir="stage2_dept_no_other_xlmr",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

stage2_trainer = Trainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=s2_train_ds,
    eval_dataset=s2_eval_ds,
    tokenizer=s2_tokenizer,
    data_collator=s2_collator,
    compute_metrics=s2_compute_metrics,
)

stage2_trainer.train()


Map:   0%|          | 0/9092 [00:00<?, ? examples/s]

Map:   0%|          | 0/1011 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.114073,0.974283,0.622796
2,0.322300,0.056027,0.989120,0.905508
3,0.322300,0.052901,0.992087,0.956310
4,0.041100,0.031842,0.994065,0.957675


TrainOutput(global_step=1140, training_loss=0.1611395579681062, metrics={'train_runtime': 221.796, 'train_samples_per_second': 163.97, 'train_steps_per_second': 5.14, 'total_flos': 312791621773344.0, 'train_loss': 0.1611395579681062, 'epoch': 4.0})

#####Eval

In [ ]:
# --- predict ---

X_test = df_t_annotated["position"]
y_test = df_t_annotated["department"].to_numpy()

# --- Stage 1 predictions on linkedin ---
test_s1 = Dataset.from_pandas(pd.DataFrame({"text": X_test}).reset_index(drop=True))
test_s1 = test_s1.map(s1_tokenize, batched=True, remove_columns=["text"])
pred_s1 = stage1_trainer.predict(test_s1)
stage1_pred_is_other = np.argmax(pred_s1.predictions, axis=-1).astype(int)

y_test_is_other = (y_test == OTHER).astype(int)

print("Stage 1 accuracy (Other vs Rest):", accuracy_score(y_test_is_other, stage1_pred_is_other))
print(classification_report(y_test_is_other, stage1_pred_is_other, target_names=["Not-Other", "Other"], zero_division=0))

# --- Stage 2 predictions on linkedin ---
test_s2 = Dataset.from_pandas(pd.DataFrame({"text": X_test}).reset_index(drop=True))
test_s2 = test_s2.map(s2_tokenize, batched=True, remove_columns=["text"])
pred_s2 = stage2_trainer.predict(test_s2)
stage2_pred_ids = np.argmax(pred_s2.predictions, axis=-1)
stage2_pred_dept = np.array([dept_id2label[i] for i in stage2_pred_ids], dtype=object)

mask_test_not_other = (y_test != OTHER)
print("Stage 2 accuracy (on not-Other rows):",
      accuracy_score(y_test[mask_test_not_other], stage2_pred_dept[mask_test_not_other]))
print(classification_report(y_test[mask_test_not_other], stage2_pred_dept[mask_test_not_other], zero_division=0))

# --- Final combined prediction ---
final_pred_dept = np.where(stage1_pred_is_other == 1, OTHER, stage2_pred_dept)

print("Overall accuracy:", accuracy_score(y_test, final_pred_dept))
print(classification_report(y_test, final_pred_dept, zero_division=0))

labels = sorted(set(y_test) | set(final_pred_dept))
cm = confusion_matrix(y_test, final_pred_dept, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Map:   0%|          | 0/2638 [00:00<?, ? examples/s]

Stage 1 accuracy (Other vs Rest): 0.5303260045489007
              precision    recall  f1-score   support

   Not-Other       0.53      1.00      0.69      1386
       Other       0.78      0.01      0.03      1252

    accuracy                           0.53      2638
   macro avg       0.66      0.51      0.36      2638
weighted avg       0.65      0.53      0.38      2638



Map:   0%|          | 0/2638 [00:00<?, ? examples/s]

Stage 2 accuracy (on not-Other rows): 0.6479076479076479
                        precision    recall  f1-score   support

        Administrative       0.34      0.18      0.23        84
  Business Development       0.51      0.46      0.48        78
            Consulting       0.83      0.57      0.68       195
      Customer Support       0.85      0.35      0.50        48
       Human Resources       0.73      0.59      0.65        70
Information Technology       0.51      0.94      0.66       312
             Marketing       0.74      0.43      0.54       133
    Project Management       0.80      0.70      0.74       175
            Purchasing       0.48      0.56      0.51        72
                 Sales       0.93      0.76      0.84       219

              accuracy                           0.65      1386
             macro avg       0.67      0.55      0.58      1386
          weighted avg       0.69      0.65      0.64      1386

Overall accuracy: 0.3464746019711903
       

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,15,2,1,0,2,48,2,2,2,10,0
Business Development,1,36,5,0,11,17,4,0,0,0,4
Consulting,4,18,111,1,0,39,4,0,5,11,2
Customer Support,2,3,0,17,0,22,0,0,0,3,1
Human Resources,9,0,3,0,40,11,3,1,0,2,1
Information Technology,1,7,3,2,0,291,0,1,7,0,0
Marketing,7,1,3,0,0,46,57,1,13,4,1
Other,89,21,88,4,35,700,63,18,60,141,33
Project Management,0,4,3,0,0,40,3,0,122,1,2


##Seniority models

####train/Test split - seniority-v2

In [ ]:
MODEL = "xlm-roberta-base"
MAX_LEN = 48
BATCH = 32
EPOCHS = 3
set_seed(42)

# --- Labels ---
sen_labels = sorted(df_seniority["label"].unique().tolist())
sen_label2id = {l: i for i, l in enumerate(sen_labels)}
sen_id2label = {i: l for l, i in sen_label2id.items()}

# --- train/eval data ---
df_sen = df_seniority.copy()
df_sen["labels"] = df_sen["label"].map(sen_label2id)

train_df, eval_df = train_test_split(
    df_sen[["text", "labels"]],
    test_size=0.10,
    random_state=42,
    stratify=df_sen["labels"],
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds  = Dataset.from_pandas(eval_df.reset_index(drop=True))

# --- Tokenize  ---
tokenizer = AutoTokenizer.from_pretrained(MODEL)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_ds  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

# --- Metrics ---
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(sen_labels),
    label2id=sen_label2id,
    id2label=sen_id2label,
)

args = TrainingArguments(
    output_dir="sen_model_xlm-r_2",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer_sen_model_xlmr_2 = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_sen_model_xlmr_2.train()

Map:   0%|          | 0/8485 [00:00<?, ? examples/s]

Map:   0%|          | 0/943 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.114287,0.974549,0.943723
2,0.344400,0.055547,0.990456,0.985758
3,0.344400,0.050144,0.990456,0.984735


TrainOutput(global_step=798, training_loss=0.23119931471975227, metrics={'train_runtime': 136.1072, 'train_samples_per_second': 187.022, 'train_steps_per_second': 5.863, 'total_flos': 220782844463640.0, 'train_loss': 0.23119931471975227, 'epoch': 3.0})

####Threshold to predict "Professional"

In [ ]:
THRESHOLD = 0.55

ext_df = df_t_annotated.copy()
ext_df["text"] = ext_df["position"]
ext_ds = Dataset.from_pandas(ext_df[["text"]].reset_index(drop=True), preserve_index=False)
ext_ds = ext_ds.map(tokenize, batched=True, remove_columns=["text"])

# --- predict ---
pred = trainer_sen_model_xlmr_2.predict(ext_ds)
logits = torch.tensor(pred.predictions)
probs = torch.softmax(logits, dim=-1).numpy()

max_prob = probs.max(axis=1)
pred_id  = probs.argmax(axis=1)

y_true = df_t_annotated["seniority"].to_numpy()
y_pred = np.array([sen_id2label[int(i)] for i in pred_id], dtype=object)

print("Without threshold:")
print("accuracy:", round((y_pred == y_true).mean(), 4))
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
display(cm_df)

y_pred[max_prob < THRESHOLD] = "Professional"


print("With threshold:")
print("accuracy:", round((y_pred == y_true).mean(), 4))
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Map:   0%|          | 0/2638 [00:00<?, ? examples/s]

Without threshold:
accuracy: 0.3772
              precision    recall  f1-score   support

    Director     0.7267    0.8865    0.7987       141
      Junior     0.1905    0.6261    0.2921       230
        Lead     0.7942    0.5957    0.6807       460
  Management     0.6495    0.7225    0.6840       418
Professional     0.0000    0.0000    0.0000      1219
      Senior     0.1667    0.8824    0.2804       170

    accuracy                         0.3772      2638
   macro avg     0.4213    0.6189    0.4560      2638
weighted avg     0.3076    0.3772    0.3133      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,2,0,7,0,7
Junior,1,144,4,8,0,73
Lead,2,48,274,33,0,103
Management,41,18,10,302,0,47
Professional,1,535,55,108,0,520
Senior,2,9,2,7,0,150


With threshold:
accuracy: 0.3901
              precision    recall  f1-score   support

    Director     0.7267    0.8865    0.7987       141
      Junior     0.1951    0.6261    0.2975       230
        Lead     0.8006    0.5935    0.6816       460
  Management     0.6726    0.7225    0.6967       418
Professional     0.6863    0.0287    0.0551      1219
      Senior     0.1691    0.8824    0.2838       170

    accuracy                         0.3901      2638
   macro avg     0.5417    0.6233    0.4689      2638
weighted avg     0.6301    0.3901    0.3416      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,2,0,7,0,7
Junior,1,144,4,8,5,68
Lead,2,42,273,33,8,102
Management,41,16,10,302,2,47
Professional,1,526,52,92,35,513
Senior,2,8,2,7,1,150


#####find best THRESHOLD

In [ ]:
pred = trainer_sen_model_xlmr_2.predict(ext_ds)
probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()

best_p = probs.max(axis=1)
pred_id = probs.argmax(axis=1)

base_pred = np.array([sen_id2label[int(i)] for i in pred_id], dtype=object)
y_true = df_t_annotated["seniority"].to_numpy()

best_t = None
best_score = -1

for THRESHOLD in np.arange(0.30, 0.95, 0.05):
    y_pred = base_pred.copy()
    y_pred[best_p < THRESHOLD] = "Professional"

    score = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    print(f"t={THRESHOLD:.2f}  macro_f1={score:.4f}  acc={acc:.4f}")

    if score > best_score:
        best_score = score
        best_t = THRESHOLD

t=0.30  macro_f1=0.4560  acc=0.3772
t=0.35  macro_f1=0.4560  acc=0.3772
t=0.40  macro_f1=0.4564  acc=0.3776
t=0.45  macro_f1=0.4568  acc=0.3779
t=0.50  macro_f1=0.4615  acc=0.3829
t=0.55  macro_f1=0.4689  acc=0.3901
t=0.60  macro_f1=0.4748  acc=0.3969
t=0.65  macro_f1=0.4811  acc=0.4037
t=0.70  macro_f1=0.4886  acc=0.4124
t=0.75  macro_f1=0.4970  acc=0.4223
t=0.80  macro_f1=0.5052  acc=0.4333
t=0.85  macro_f1=0.5149  acc=0.4454
t=0.90  macro_f1=0.5292  acc=0.4663


#####without "Professional" in evaluation data

In [ ]:
# --- predict ---
pred = trainer_sen_model_xlmr_2.predict(ext_ds)
y_pred_all = np.argmax(pred.predictions, axis=-1)

y_true_str = df_t_annotated["seniority"].astype(str).to_numpy()
keep = (y_true_str != "Professional")

print("n_total:", len(y_true_str), "| dropped_professional:", int((~keep).sum()), "| n_eval:", int(keep.sum()))

y_true = np.array([sen_label2id[s] for s in y_true_str[keep]], dtype=int)
y_pred = y_pred_all[keep]

print("accuracy:", round((y_pred == y_true).mean(), 4))

labels = sorted(set(y_true) | set(y_pred))
target_names = [sen_id2label[int(i)] for i in labels]

print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(target_names, name="true"), columns=pd.Index(target_names, name="pred"))
cm_df


n_total: 2638 | dropped_professional: 1219 | n_eval: 1419
accuracy: 0.7012
              precision    recall  f1-score   support

    Director     0.7310    0.8865    0.8013       141
      Junior     0.6516    0.6261    0.6386       230
        Lead     0.9448    0.5957    0.7307       460
  Management     0.8459    0.7225    0.7794       418
      Senior     0.3947    0.8824    0.5455       170

    accuracy                         0.7012      1419
   macro avg     0.7136    0.7426    0.6991      1419
weighted avg     0.7810    0.7012    0.7149      1419



pred,Director,Junior,Lead,Management,Senior
true,,,,,
Director,125,2,0,7,7
Junior,1,144,4,8,73
Lead,2,48,274,33,103
Management,41,18,10,302,47
Senior,2,9,2,7,150
